# Custom Chatbot Project

I chose the provided 2023_fashion_trends csv file as my dataset. I think it makes sense because it occured after the pretraining of gpt3.5, so it is the perfect use case of providing additional context using RAG technique.

## Data Wrangling

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('2023_fashion_trends.csv')

In [3]:
df = df.drop(columns=["URL", "Source"])

In [4]:
df = df.rename(columns={ "Trends": "text" })

In [5]:
# Remove '2023 Fashion Trend:' prefix because every row of data is already describing a trend. I think if we don't remove them, more weight
# Could be given to those rows

df["text"] = df["text"].apply(lambda x: x.replace("2023 Fashion Trend:", ""))

## Custom Query Completion

In [6]:
import openai

In [7]:
openai.api_key = ""
openai.api_base = "https://openai.vocareum.com/v1"

In [8]:
EMBEDDING_MODEL = "text-embedding-3-small"
response = openai.Embedding.create(
    input=df["text"].to_list(),
    model=EMBEDDING_MODEL
)

In [9]:
df["embedding"] = [row["embedding"] for row in response.data]

In [10]:
import tiktoken

# Can't use encoding_for_model in the version of tiktoken
encoder = tiktoken.get_encoding("cl100k_base")
df["token"] = df["text"].apply(lambda x: encoder.encode(x))

In [11]:
from openai.embeddings_utils import distances_from_embeddings

def get_sorted_cosine_distance(prompt):

    embedding = openai.Embedding.create(
        input=[prompt],
        model=EMBEDDING_MODEL
    )["data"][0]["embedding"]
        
    distances = [(i, distance) for i, distance in enumerate(distances_from_embeddings(embedding, df["embedding"].to_list()))]
    distances.sort(key=lambda x: x[1])

    return distances
    

In [12]:
def adjust_prompt(prompt, should_inject_context=True):
    MAX_TOKENS = 4096 * .8 # let's keep some room for the output tokens

    TEMPLATE = """
You are a helpful chatbot that answers Q&A Questions. You are an expert in fashion.
Your job is to take questions and output answers. One example is:
- Question: What decade were bell bottom jeans in style?
- Answer: Bell bottom jeans were in style in the 1970s

{context}

## Question:
{prompt}
"""

    if not should_inject_context:
        return TEMPLATE.format(context="", prompt=prompt)
    
    
    token_count = len(encoder.encode(TEMPLATE)) + len(encoder.encode(prompt))
    additional_context = []
    distances = get_sorted_cosine_distance(prompt)
    for i, distance in distances:
        row = df.iloc[i]
        token_count += len(row["token"])
        if token_count >= MAX_TOKENS:
            break
        additional_context.append(row["text"])

    context = "## Additional Context:\n" + ('\n-').join(additional_context)
    return TEMPLATE.format(context=context, prompt=prompt)
        

In [13]:
def complete(prompt, should_inject_context=True):
    text = adjust_prompt(prompt, should_inject_context)
    token_count = len(encoder.encode(text))
    
    response = openai.Completion.create(
        model="gpt-3.5-turbo-instruct",
        prompt=text,
        max_tokens=4096 - token_count
    )

    return response["choices"][0]["text"]

In [14]:
df

,text,embedding,token
0,Red. Glossy red hues took over the Fall 2023 ...,"[0.009354429319500923, 0.028719132766127586, 0...","[3816, 13, 67142, 88, 2579, 82757, 3952, 927, ..."
1,"Cargo Pants. Utilitarian wear is in for 2023,...","[0.011290382593870163, 0.037782032042741776, 0...","[62388, 67553, 13, 10377, 20631, 10051, 374, 3..."
2,"Sheer Clothing. ""Bare it all"" has been the mo...","[0.0930228978395462, 0.013377992436289787, 0.0...","[3005, 261, 54758, 13, 330, 33, 548, 433, 682,..."
3,Denim Reimagined. From double-waisted jeans t...,"[0.05959445238113403, 0.0383843258023262, 0.00...","[9973, 318, 1050, 29116, 1619, 13, 5659, 2033,..."
4,Shine For The Daytime. The amount of shine on...,"[0.0248839370906353, -0.016262058168649673, -0...","[87255, 1789, 578, 6187, 1712, 13, 578, 3392, ..."
...,...,...,...
77,"If lime green isn't your vibe, rest assured th...","[0.054094839841127396, 0.003149157389998436, 0...","[2746, 42819, 6307, 4536, 956, 701, 47811, 11,..."
78,"""As someone who can clearly (not fondly) remem...","[0.03556917607784271, -0.04244769364595413, 0....","[48240, 4423, 889, 649, 9539, 320, 1962, 21901..."
79,"""Combine this design shift with the fact that ...","[0.07606058567762375, 0.005213485565036535, -0...","[1, 82214, 420, 2955, 6541, 449, 279, 2144, 43..."
80,Thought party season ended at the stroke of mi...,"[0.019862964749336243, 0.042307399213314056, -...","[85269, 4717, 3280, 9670, 520, 279, 12943, 315..."


## Custom Performance Demonstration

TODO: In the cells below, demonstrate the performance of your custom query using at least 2 questions. For each question, show the answer from a basic `Completion` model query as well as the answer from your custom query.

### Question 1

In [15]:
# With Rag
print(complete("What were the top 3 trends in fashion during the summer of 2023 in America?"))


Answer: The top three trends for the summer of 2023 in America were cargo pants, sheer clothing, and elevated basics.


In [16]:
# Without RAG
print(complete("What were the top 3 trends in fashion during the summer of 2023 in America?", False))


Answer: As a chatbot, I only have access to current and historical fashion trends. Predictions for fashion trends in 2023 are not available at this time. Can I assist with any other questions?


### Question 2

In [17]:
# With Rag
print(complete("Based on the fashion trends of 2023, what do you predict may be some trends in 2024?"))

## Answer:
While it is hard to predict with certainty, the fashion trends of 2023 may serve as a good indication of what to expect in 2024. Some possible trends could include a continuation of elevated basics, sheer clothing, denim-on-denim looks, and draping. We may also see a resurgence of cargo pants, tailored pieces, and pinstripe designs. Bold prints, like florals, and a mix of textures, such as mesh and metallics, may also be popular. Additionally, the influence of the 90s and early 2000s may continue to be seen in fashion.


In [18]:
# Without Rag
print(complete("Based on the fashion trends of 2023, what do you predict may be some trends in 2024?", False))


Answer: It is difficult to predict exactly what fashion trends will be popular in 2024, as fashion is constantly evolving. However, based on current trends, some potential predictions for 2024 could be bold and colorful prints, sustainable and eco-friendly fashion, and statement accessories such as oversized hats or chunky jewelry. It is also possible that past fashion trends from the 1980s or 1990s may make a comeback in 2024. Ultimately, the best way to stay on top of fashion trends is to follow fashion influencers and keep an eye on fashion shows and magazines as they showcase upcoming trends.


## Now your turn!

In [ ]:
# With Rag
print(complete(input()))

In [ ]:
# Without Rag
print(complete(input(), False))